# Biological Oceanography - Week 6 Practical (Phytoplankton Blooms)

## Overview of the Practical

In this practical we will learn how to process data from a Biogeochemical Argo (BGC-Argo) float. In your groups, you will:

1. Choose a region of interest with an available BGC-Argo float.
2. Download BGC-Argo data for your chosen float and upload them to the jupyter notebook.
3. Run through a series of processing steps, finishing by plotting a time-series of physical (temperature and salinity), and biogeochemical (chlorophyll-a concentration, particulate backscattering and dissolved oxygen) data.
4. Determine the likely drivers of phytoplankton dynamics in your region using the physical and biogeochemical data, and what you've learned in the Biological Oceanography course so far.
5. Sketch how nutrient concentration (nitrate/phosphate/iron) within the mixed layer may vary across your time-series.
6. Summarise your thinking in **one PowerPoint slide** and a **three-minute presentation** to feed back to your peers at the end of today's session. You do not all have to talk but you do all need to contribute to the analysis and thinking.

For a detailed guide on BGC-Argo data, please refer to Bittig et al. (2019, https://doi.org/10.3389/fmars.2019.00502).

This practical is based on a notebook originally written by Bob Brewin (University of Exeter) and modified by Noam Vogt-Vincent and Heather Bouman.

## Technical instructions

Please note that this practical is designed such that each cell (block of Python code) is run after the previous cell. If you clear or restart the practical, you will need to run through each cell from the start. If you are running this practical through binder, please be aware that the memory can sometimes get randomly cleared, resulting in error messages claiming variables do not exist. If this occurs, you just need to rerun the cells again (e.g. `Run > Run All Cells`). If you are using binder, please also note that your session **will not be saved**! If you want to save your notebook and/or figures for future use, you will need to make an offline copy (`File > Save Notebook As`).

Please run the following lines of code to ensure you have the right packages installed for the practical. To do this click on the code (cell) below and then click the run button at the top of the page.

In [ ]:
#Import all packages required for practical
import numpy as np
import numpy.ma as ma
import math as mt
import julian
import datetime
import scipy
import matplotlib as mpl
import netCDF4 as nc
import pandas as pd
import gsw
from math import nan
from netCDF4 import Dataset
from matplotlib import pyplot as plt 
from matplotlib.colors import LogNorm, Normalize
from matplotlib.cm import ScalarMappable
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
from scipy import interpolate
from scipy import signal
from bloom_plots import *

## Part 1: Selecting a BGC-Argo float and region of interest

In total, 5 regions have been selected for this practical, and a total of 10 BGC-Argo floats. They are:

**Group 1: High Latitude North Atlantic**
- 6901523 (Coriolis)
- 6901515 (Coriolis)

**Group 2: Mediterranean**
- 6901528 (Coriolis)
- 6901648 (Coriolis)

**Group 3: Southern Ocean**
- 6901004 (Coriolis)
- 6901585 (Coriolis)

**Group 4: Polar Seas**
- 5904472 (AOML)
- 5907058 (AOML)

**Group 5: Low Latitude Pacific**
- 6901658 (Coriolis)
- 5906529 (AOML)

**Group 6: South Atlantic**
- 4903459 (AOML)
- 6901582 (Coriolis)

The names in (brackets) refer to the hosting data centre, which you will need in Part 2 below.

## Part 2: Viewing and downloading BGC-Argo float data

### Oceanographic Autonomous Observations (OAO) portal

A nice visualisation portal for viewing BGC-Argo float data can be found at the Oceanographic Autonomous Observations (OAO) virtual platform based in Villefranche-sur-mer (France). Please click on [this link](https://maps.biogeochemical-argo.com/bgcargo/).

You will see the locations of BGC-Argo float data in different regions of the ocean. It is worth noting that not all available BGC-Argo floats are on this webpage. Nonetheless, it is a useful webpage for visualising many of the BGC-Argo floats in the ocean. Familiarise yourself with the webpage and note the following features.

- Float WMO numbers are on the left-hand side bar.
- The toggle switches at the top allow you to focus in on floats and tracks.
- Float status is also shown on the map according to colour, where: green indicates data that was last communicated within 12 days; and grey floats refer to floats that last communicated data over 30 days ago, and thus likely to have reached the end of their lifespan.
- The drop down on the top-right allows you to change the background ocean image.
- The scroll bar at the bottom of the page allows you to go back in time.

### Exploring BGC-Argo float data on the OAO portal

Every BGC-float has a WMO number (that stands for the World Meteorological Organization) on the left hand panel. It is useful to note this number down when downloading float data, as it provides a unique identifying number. If you click on a float WMO number on the left side of the OAO portal some additional information appears:

- At the bottom of the left-hand panel on the webpage some links are provided to information about the float. Note there might be some small differences between this info and the float files we use in this practical, as float data are often reprocessed, and info updated.
- If you click on a float WMO number, and then click on one profile (dot on the float path) and an additional webpage will pop up (see figure above) showing data from the float and what variables it includes. In this practical we will use floats with data for temperature, salinity, dissolved oxygen, chlorophyll-a (Chl-a) and backscattering by particles at 700 nm.



### Downloading BGC-Argo float data

Bittig et al. (2019, https://doi.org/10.3389/fmars.2019.00502) describe (see their Table 5) the different levels of BGC-Argo data available to download. Typically, they are "real-time", "delayed-mode" and "all-cycle" data. Data is available for core-Argo variables (c-files), biogeochemical Argo variables (b-files) and integrated variables data (s-files). Data files are provided in netCDF format, which is a common format used in climate science for storing multidimensional data.  
In this practical we will use "all-cycle" s-files, defined by float number (e.g. 6901437_Sprof.nc). These files include all profiles from a BGC-Argo float, that generally have been controlled to a certain level for data quality (e.g. for example we will use the Chl-a adjusted profiles that include a quenching correction) and the b-file data have been adjusted to the same depth (pressure) as the c-file data, making it easier to work on both core-Argo and biogeochemical Argo variables. Data is downloaded through FTP servers. 

Please follow these steps to download your data:

1. Find the WMO float number of interest. You can use the OAO portal as a guide for some floats (e.g. 6901437)
2. Connect to a [BGC-Argo server](https://data-argo.ifremer.fr/dac/). There are 11 data centres but not all floats are in all data centres. Refer back to section 1 to see which data centre has your float.
4. Find the folder with the float number and download the s-files (e.g. 6901437_Sprof.nc). Note not all floats have s-file data. 

In this practical, we will be using the following variables: Pressure (dBar, we assume equivalent to depth in meters); Temperature ($^{o}C$); Salinity (PSU); Dissolved Oxygen (micro mol kg$^{-3}$); Chlorophyll-a (mg m$^{-3}$); and backscattering by particles (at wavelength of 700 nm, m$^{-1}$). These floats have all operated for 300 days or longer, and thus provide data on a full seasonal cycle (approximately) or longer. More variables are also available within these files, but they will not be used in this practical. Please refer to the Bittig et al. (2019) paper to learn more.

### Importing BGC-Argo float data into Jupyter Lab
Once you have downloaded your BGC-Argo data (which, as a reminder, should have a file name along the lines of `XXXXXXX_Sprof.nc`), you need to import it into your Jupyter session. If you are using your own local Python installation, this is easy - just use the file browser on the left of your screen to find the file and/or use the full file path when importing it. 

If you are on binder, you will need to first upload it to your session. In the file browser on the left of your screen, make sure you're in the same directory as the jupyter notebook file, and then press `Upload Files` (the up arrow just above the file browser). After uploading the file, it should appear in the file browser alongside the jupyter notebook file.

Now, insert the correct float number for your region in the code block below, and run it.

In [ ]:
#Float Number (change as required): 
FLOAT_NUMBER = 'INSERT_NUMBER_HERE'

### Reading BGC-Argo float data using Python

Having selected the BGC-Argo float we then need to read the data using Python. 

The first thing we will do is find out the type of BGC-Argo float we are using, the Principal Investigator (PI) in charge of it, and the project it was develop for. Run the following lines of code. **If you get an error here, it is probably because your BGC-Argo file is in the wrong place.**

In [ ]:
#Read the BGC-Argo netCDF file into Python
BGC_DATA     = FLOAT_NUMBER+'_Sprof.nc'
fh           = Dataset(BGC_DATA, mode='r')

#Read key information on the float
PROJECT_NAME  = fh.variables['PROJECT_NAME'][:]
PI_NAME       = fh.variables['PI_NAME'][:]
PLATFORM_TYPE = fh.variables['PLATFORM_TYPE'][:]
CYCLE_NUMBER  = fh.variables['CYCLE_NUMBER'][:]

print('Project Name =', str(PROJECT_NAME[0].tobytes(), 'utf-8'))
print('PI Name =', str(PI_NAME[0].tobytes(), 'utf-8'))
print('Platform Type =', str(PLATFORM_TYPE[0].tobytes(), 'utf-8'))
print('Maximum Cycle Number =', str(np.max(CYCLE_NUMBER)))

The code outputs the Project Name, PI name, Platform Type and maximum cycle number (i.e. number of profiles). The Project Name and PI name is important as they both should be appropriately recognised should we use the data (e.g. for a scientific publication), or contacted if there are any issues. If ARGO data are used for scientific publication, you should always acknowledge the ARGO programme too (see https://argo.ucsd.edu/data/acknowledging-argo/). The Platform Type provides the type of BGC-Argo float we are using. Information on the type of float can be found here https://argo.ucsd.edu/how-do-floats-work/float-types/ and may be useful for your coursework.

Having determined the Project Name, PI name, Platform Type and maximum cycle number, next we will read in the variables of interest into Python and convert the time variable from Julian day relative to a reference time, to a format we are more familiar with. The next cell of code does the following

1. Reads the variables of interest from the netCDF file (note the specific names of the variables that need to be read in).
2. Calculates density from temperature and salinity, and computes the MLD as $\sigma(z_{\textrm{MLD}})=\sigma(10\ \textrm{dbar})+0.03\ \textrm{kg m}^{-3}$.
3. Use Python functions to convert the time variable from Julian day to Year-Month-Day, useful when plotting a time-series.
4. Create a matrix of time (for every depth variable) that will be used later for the contour plot.

Run the following lines of code. 

In [ ]:
#Read time and locations of profiles 
JULD         = fh.variables['JULD'][:]            #Julian day of the profile.
LONGITUDE    = fh.variables['LONGITUDE'][:]       #Longitude degrees East
LATITUDE     = fh.variables['LATITUDE'][:]        #Latitude degress North

#Read netCDF CORE ARGO
PRESSURE     = fh.variables['PRES'][:]            #Pressure (Dbar) we assume equivalent to depth (m)
TEMP         = fh.variables['TEMP'][:]            #Temperature (degress C)
PSAL         = fh.variables['PSAL'][:]            #Salinity (PSU)

#Read netCDF BGC ARGO ADJUSTED (s-profile)
CHLA         = fh.variables['CHLA_ADJUSTED'][:]   #Chlorophyll-a 
BBP          = fh.variables['BBP700_ADJUSTED'][:] #Backscattering at 700nm (m^-1)
DISS_OXY     = fh.variables['DOXY_ADJUSTED'][:]   #Dissolved Oxygen (micro mole kg^-3)
fh.close()

#Convert time from Julian day to Year-Month-Day T HOUR:MINUTE:SECOND
REFERENCE_DATE = 2433282.50000  #JULIAN DATE for reference 1950-01-01 00:00:00
TIME_BGC       = np.empty(len(JULD), dtype='datetime64[s]')
MONTH_BGC      = np.empty(len(JULD), dtype='datetime64[s]')
for i in range(len(JULD)):
    TIME_BGC[i] = julian.from_jd(JULD[i]+REFERENCE_DATE, fmt='jd')
    
#Create a time matrix needed later for the contour plot (providing time for each depth level)
TIME_BGC_MATRIX = np.empty([len(TEMP[:,0]),len(TEMP[0,:])], dtype='datetime64[s]')
for i in range(len(TEMP[0,:])):
    TIME_BGC_MATRIX[:,i] = TIME_BGC

# Calculate potential density and mixed-layer depth
# MLD is the depth where potential density exceeds the
# density at 10 dbar by 0.03 kg/m3.
# A median filter reduces isolated density spikes, and the
# threshold must remain exceeded for at least 10 dbar.

MLD_THRESHOLD = 0.03       # Density threshold (kg/m3)
MLD_REFERENCE = 10         # Reference pressure (dbar)
MLD_MEDIAN_WINDOW = 5      # Median filter width (dbar)
MLD_PERSISTENCE = 10       # Required threshold exceedance (dbar)
MLD_MAX_GAP = 20           # Maximum interpolation gap (dbar)

DENSITY, MLD = calculate_density_mld(
    PRESSURE, TEMP, PSAL, LONGITUDE, LATITUDE,
    threshold=MLD_THRESHOLD,
    reference=MLD_REFERENCE,
    median_window=MLD_MEDIAN_WINDOW,
    persistence=MLD_PERSISTENCE,
    max_gap=MLD_MAX_GAP
)

print("Density range:", np.nanmin(DENSITY), np.nanmax(DENSITY))
print("Valid MLD estimates:", np.isfinite(MLD).sum(), "of", len(MLD))


## Part 3: Plotting BGC-Argo float data using Python

### Latitude and longitude plot

We have now read the data from the netCDF file. First, let's produce a plot of the locations of the float profiles from the start of its mission to the end. Run the following lines of code. 

In [ ]:
# FIGURE 1 — Float trajectory and global context
# Change FIGSIZE to change the physical dimensions of the whole figure.

FIGSIZE = (10.2, 7.2)  # (width, height) in inches
plot_float_trajectory(
    dict(LONGITUDE=LONGITUDE, LATITUDE=LATITUDE,
         TIME_BGC=TIME_BGC, FLOAT_NUMBER=FLOAT_NUMBER),
    dict(figsize=FIGSIZE)
)

Figure 1 above shows the track of the float. You can compare it with the tracks on the OAO portal for the same float to cross-check. The plot shows the extent to which the float drifted during its lifespan. Once you are happy with it, you can right click and save the image to a safe location.

**Question: Based on what you know about the physical oceanography of your region, does its trajectory make sense?**


### Profile plots

Next, let's plot vertical data from a single profile from the float to see how the variables change with depth.

In [ ]:
# FIGURE 2 — Single vertical profile
# Change CYCLE_TO_PLOT to select a different *Argo cycle number* (not an array index).
# MAX_DEPTH controls the depth displayed, in metres (approximately dbar).

CYCLE_TO_PLOT = 1
MAX_DEPTH = 200
FIGSIZE = (16, 7)  # Optional: (width, height) in inches
plot_vertical_profile(
    dict(CYCLE_NUMBER=CYCLE_NUMBER, TEMP=TEMP, PSAL=PSAL,
         DISS_OXY=DISS_OXY, CHLA=CHLA, BBP=BBP, PRESSURE=PRESSURE,
         TIME_BGC=TIME_BGC, LONGITUDE=LONGITUDE, LATITUDE=LATITUDE,
         FLOAT_NUMBER=FLOAT_NUMBER),
    dict(INDEX_PROFILE=CYCLE_TO_PLOT, max_depth=MAX_DEPTH, figsize=FIGSIZE)
)

Figure 2 above shows a vertical profile of variables from the float in the top 200m for a given cycle number. You can change the profile by changing the profile number in the code (under `CYCLE_TO_PLOT`). Note the number has to be between the minimum and maximum number of float cycles. 

**Question: For a few profiles, identify the mixed layer depth and interpret the physical/biological drivers of the vertical patterns you can see.**

**Question: PAR is not available here. However, how might you be able to calculate the photic depth based on available measurements?**

You can modify the presentation of the figure by changing the parameters at the top of the code, most importantly `CYCLE_TO_PLOT` and `MAX_DEPTH`. Try to make the plot your own, and if you feel confident about changing other parts you can but be careful not to break the code. Once you are happy with it, you can right click and save the image to a safe location. You could produce a few profiles at different stages of the float’s life. 

### Depth-time plots

Whereas these individual profiles are nice to see, viewing the data from all profiles in one plot is very useful for understanding how the data changes over the lifetime of the float, and understanding seasonal cycles. To do this we will produce a depth-time plot of the BGC-Argo float data.

While the s-files provide each variable on the same pressure (depth) axis, some sensors sample at a higher frequency than others. This results in gaps in data for some variables. For the purposes of the plot, we will interpolate (linearly) the data for each variable, such that we have a value for each variable on every pressure level (barring gaps exceeding 20 dbar). The advantage of doing this is that we can produce a nice 2D plot, with data for every depth. The disadvantage is that it can have the potential to introduce artifacts onto the plot, for example, from interpolating data with only a few measurements over large depth ranges.

The following lines of code loop through each profile and use the interpolate function in Python to fill data in the missing depth bins. Note for the backscattering variable, we also use a median filter (11 point) to remove spikes in the data (for cases where we have more than 11 data points) as we are interested in the smoothed fields (though the spikes can provide very useful information on large particles, see Briggs et al. 2020 https://doi.org/10.1126/science.aay1790)

Run the following lines of code to perform the interpolation.

In [ ]:
# Interpolate measured variables onto each profile's observed pressure levels.
# Valid pressure/value pairs only; missing values and large vertical gaps stay blank.

MAX_VERTICAL_GAP_INTERPOLATION = 20  # Largest interpolated depth gap (dbar)
TEMP_INT, PSAL_INT, CHLA_INT, BBP_INT, DISS_OXY_INT = interpolate_original_profiles(
    dict(TEMP=TEMP, PSAL=PSAL, CHLA=CHLA, BBP=BBP,
         DISS_OXY=DISS_OXY, PRESSURE=PRESSURE, JULD=JULD),
    max_vertical_gap=MAX_VERTICAL_GAP_INTERPOLATION
)

Having interpolated the data, next we will produce a depth-time plot. Here we will produce a single figure with five subplots, all on the same time and depth axis.

In [ ]:
# FIGURE 3 — Seasonal time–depth sections
# Scientific choices: display depth, depth spacing, and maximum gaps allowed.
# Large gaps are LEFT BLANK, not filled by invented observations.
# Percentiles control only colour scaling (not the data themselves).

MAX_DEPTH = 200             # Depth of section (m, approximately dbar)
DEPTH_STEP = 2              # Vertical grid spacing (dbar)
MAX_VERTICAL_GAP = 20       # Do not interpolate between levels farther apart (dbar)
MAX_TEMPORAL_GAP_DAYS = 45  # Mask intervals between widely separated profiles (days)
COLOUR_PERCENTILES = (1, 99)  # Lower and upper colour-scale percentiles (%)
FIGSIZE = (13.5, 13.7)     # Optional: (width, height) in inches

plot_bgc_timeseries(
    dict(TIME_BGC=TIME_BGC, LONGITUDE=LONGITUDE, LATITUDE=LATITUDE,
         PRESSURE=PRESSURE, MLD=MLD, TEMP=TEMP, PSAL=PSAL,
         DISS_OXY=DISS_OXY, CHLA=CHLA, BBP=BBP, FLOAT_NUMBER=FLOAT_NUMBER),
    dict(MAX_DEPTH=MAX_DEPTH, DEPTH_STEP=DEPTH_STEP,
         MAX_VERTICAL_GAP=MAX_VERTICAL_GAP,
         MAX_TEMPORAL_GAP_DAYS=MAX_TEMPORAL_GAP_DAYS,
         Percentiles_lower=COLOUR_PERCENTILES[0],
         Percentiles_upper=COLOUR_PERCENTILES[1], figsize=FIGSIZE)
)

The above figure shows a time-series of data from the BGC-Argo float you selected in the top 200m. For convenience, the MLD computed from the T/S-derived density is overlaid onto the Chl-a plot. However, please note that we are using a very simple algorithm for calculating the MLD which is sensitive to the density measured by the float in shallow water. If the MLD line looks like it isn't consistent with the mixed layer you can diagnose from the T/S data then ignore it.

You can change components of the plot by changing the parameters at the top of the code. Like previous plots, try to make the plot your own, and if you feel confident about changing other parts you can, but be careful not to break the code. Once you are happy with it, you can right click and save the image to a safe location.

**Questions: Are there any specific depth variations in the variables? Can you see any pronounced seasonal cycles in the magnitudes of the variables? Are temporal changes possibly related to spatial changes in the float location if it drifted during its lifetime (see Figure 1, also look at the float track in the OAO platform, and play with the background layers of satellite Chlorophyll-a so see if the float drifted between high and low Chl-a regions?).**

### Seasonal monthly climatology for top 100 m

To quantify monthly values for variables over the course of a season, we will compute the average of each variable in the upper 100 m (roughly corresponding to the epipelagic layer) for each profile, and then find the average value across all profiles for each month of the year. In addition, we will compute the standard deviations of each variable, across profiles for each month, to understand the variance in the data. 

Run the following lines of code to produce a plot of month-by-month averages of variables. 

In [ ]:
# FIGURE 4: Monthly climatology of the upper 100 m
#
# Each profile is averaged over the upper 100 dbar.
# Monthly means and standard deviations are calculated
# from these profile averages across all available years.

# Parameters to modify
AVERAGING_DEPTH = 100      # Upper depth range for averaging (dbar)
DEPTH_STEP = 2             # Vertical grid spacing (dbar)
MAX_VERTICAL_GAP = 20      # Maximum interpolation gap (dbar)
FIGSIZE = (11, 14.2)      # Figure dimensions (inches)

plot_monthly_climatology(
    dict(
        TIME_BGC=TIME_BGC,
        TEMP=TEMP,
        PSAL=PSAL,
        DISS_OXY=DISS_OXY,
        CHLA=CHLA,
        BBP=BBP,
        PRESSURE=PRESSURE,
        MLD=MLD
    ),
    dict(
        AVERAGING_DEPTH=AVERAGING_DEPTH,
        DEPTH_STEP=DEPTH_STEP,
        MAX_VERTICAL_GAP=MAX_VERTICAL_GAP,
        figsize=FIGSIZE
    )
)

Figure 4 above shows monthly changes in variables from the float in the upper 100 mfor a given cycle number. 

You can modify the presentation of the figure by changing the parameters at the top of the code. Once you are happy with it, you can right click and save the image to a safe location. 

## Parts 4-6
Using the above plots and modifying them as required, as well as the regional context and what you have learned so far in the biological and physical oceanography course, identify the drivers of phytoplankton dynamics in your region. Sketch the predicted seasonal cycle for nutrients (nitrate/phosphate/iron) within the mixed layer for your region. Do you think we're in a HNLC or LNLC region? Finally, put together a single powerpoint slide within your groups that concisely answers the question "What are the drivers of phytoplankton dynamics in this region?", and prepare a 3 minute talk that you will use to feed back to the class! **Send your slide to me (noam.vogt-vincent@earth.ox.ac.uk) in good time before the presentations. Groups submitting late presentations will be used as a source of nitrate (or phosphate, for the North Atlantic) to improve productivity in a subtropical gyre.**

## References

Bittig, H.C., Maurer, T.L., Plant, J.N., Wong, A.P., Schmechtig, C., Claustre, H., Trull, T.W., Udaya Bhaskar, T.V.S., Boss, E., Dall'Olmo, G. et al. (2019) A BGC-Argo guide: Planning, deployment, data handling and usage. Frontiers in Marine Science, 6, p.502. https://doi.org/10.3389/fmars.2019.00502

Bittig, H.C., Wong, A.P., Plant, J.N., and the Coriolis Argo data management team (2020). BGC- Argo synthetic profile file processing and format on Coriolis GDAC, v1.2 http://dx.doi.org/10.13155/55637

Briggs, N., Dall’Olmo, G. and Claustre, H., 2020. Major role of particle fragmentation in regulating biological sequestration of CO2 by the oceans. Science, 367(6479), pp.791-793. https://doi.org/10.1126/science.aay1790 